# Random Forest: Original and Expanded Inputs

We compare two feature sets using the same random-forest settings and
truck-grouped folds. The expanded set reuses average/difference definitions
from Dom's notebook, together with the original RF inputs. Dom's notebook
fits logistic regression and XGBoost; these two fits both use random forest.

Each row here is a truck/rail assessment. Folds keep a truck together, but the
other truck from the same car can appear in training. Notebook 08 instead
holds out complete cars for the common comparison. Follow
[INSTALL.md](../INSTALL.md) for setup.

In [ ]:
# Shared input locations; see INSTALL.md for the Teams download layout.
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository; see INSTALL.md.")
sys.path.insert(0, str(ROOT / "src"))
from project_paths import data_directory, find_project_root, merged_file
ROOT = find_project_root(ROOT)
DATA = data_directory(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score, recall_score

# Load the supplied merged dataset from the shared data folder
final_data = pd.read_csv(merged_file(DATA))

# Check that it loaded correctly
final_data.shape

## RF with the original inputs

In [ ]:
# Make copy that will be modified for use in analysis
df = final_data.copy()

# Identify each unique physical truck
df["Truck_ID"] = (
    df["Car"].astype(str) + "_" +
    df["Truck"].astype(str)
)
# list of features that will be used for RF
features = [
    "Eq Con 3mm",
    "Flange Width (Sd)_L",
    "Flange Width (Sd)_R",
    "Flange Height (Sh)_L",
    "Flange Height (Sh)_R",
    "Max Hollow Wear_L",
    "Max Hollow Wear_R",
    "Max Hollow Wear Position_L",
    "Max Hollow Wear Position_R",
    "Flange Radius_L",
    "Flange Radius_R"
]

# Confirm that target labels agree within each truck-rail group
group_cols = ["Truck_ID", "Rail_Profile"]

assert (
    df.groupby(group_cols)["Empty_Truck_Hunted"]
      .nunique(dropna=False)
      .eq(1).all()
), "Inconsistent hunting labels within truck-rail groups"

# Aggregate measurements from both axles
truck_df = (
    df.groupby(group_cols)
      .agg(
          {**{feature: "mean" for feature in features},
           "Empty_Truck_Hunted": "first"}
      )
      .reset_index()
)

In [ ]:
# Add rail profile as a categorical predictor
X = truck_df[features + ["Rail_Profile"]].copy()
X = pd.get_dummies(
    X, columns=["Rail_Profile"], dtype=int
)

y = truck_df["Empty_Truck_Hunted"]
groups = truck_df["Truck_ID"]

print("Class distribution:")
print(y.value_counts())

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=4,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

cv = GroupKFold(n_splits=5)

# Out-of-fold predictions
y_pred = cross_val_predict(
    model,
    X,
    y,
    cv=cv,
    groups=groups
)

print(classification_report(y, y_pred, zero_division=0))
print(confusion_matrix(y, y_pred))

# Save model's values for use in visual
original_y = y.copy()
original_pred = y_pred.copy()
original_keys = truck_df.loc[
    y.index, ["Truck_ID", "Rail_Profile"]
].reset_index(drop=True)

The classification report gives separate recall values for
hunting and no hunting. The original RF detects more hunting assessments than
no-hunting assessments, so accuracy alone gives an incomplete picture. The
comparison plot below includes balanced accuracy and both recall values.

## RF with expanded wear inputs

The second fit retains the original 11 numeric inputs and adds
13 averages/differences, using several definitions also used by Dom. Its full
feature list differs from Dom's 16-summary model. It also adds measurement
types absent from the first RF, so this is a feature-set comparison rather
than an isolated test of averaging.

In [ ]:
# Create wheelset hollow wear features from the left and right wheels
final_data["Hollow_Wear_Avg"] = (
    final_data["Max Hollow Wear - AAR_L"] +
    final_data["Max Hollow Wear - AAR_R"]
) / 2

final_data["Hollow_Wear_Diff"] = abs(
    final_data["Max Hollow Wear - AAR_L"] -
    final_data["Max Hollow Wear - AAR_R"]
)

# Create wheelset qR features from the left and right wheels
final_data["qR_Avg"] = (
    final_data["qR_L"] +
    final_data["qR_R"]
) / 2

final_data["qR_Diff"] = abs(
    final_data["qR_L"] -
    final_data["qR_R"]
)

# Create wheelset flange width features from the left and right wheels
final_data["Flange_Width_Avg"] = (
    final_data["Flange Width_L"] +
    final_data["Flange Width_R"]
) / 2

final_data["Flange_Width_Diff"] = abs(
    final_data["Flange Width_L"] -
    final_data["Flange Width_R"]
)

# Create wheelset maximum flange angle features
final_data["Flange_Angle_Avg"] = (
    final_data["Max Flange Angle_L"] +
    final_data["Max Flange Angle_R"]
) / 2

final_data["Flange_Angle_Diff"] = abs(
    final_data["Max Flange Angle_L"] -
    final_data["Max Flange Angle_R"]
)

# Create wheelset averages for the main wheel wear measurements
measurements = [
    "Flange Width (Sd)",
    "Flange Height (Sh)",
    "qR",
    "Flange Width",
    "Max Hollow Wear",
    "Flange Radius",
    "Max Flange Angle",
    "Min Flange Root Radius",
    "Max Hollow Wear - AAR"
]

for measurement in measurements:
    final_data[measurement + "_Avg"] = (
        final_data[measurement + "_L"] +
        final_data[measurement + "_R"]
    ) / 2

print("Dataset shape:", final_data.shape)

In [ ]:
# Make a copy for the Random Forest analysis
# Axles 1-2 belong to Truck 1; axles 3-4 belong to Truck 2.
truck_df = final_data.copy()

truck_df["Truck_ID"] = (
    truck_df["Car"].astype(str) + "_" + truck_df["Truck"].astype(str)
)

# Original wheel geometry and wheel-rail interaction predictors
features = [
    "Eq Con 3mm",
    "Flange Width (Sd)_L", "Flange Width (Sd)_R",
    "Flange Height (Sh)_L", "Flange Height (Sh)_R",
    "Max Hollow Wear_L", "Max Hollow Wear_R",
    "Max Hollow Wear Position_L", "Max Hollow Wear Position_R",
    "Flange Radius_L", "Flange Radius_R", 
    "Hollow_Wear_Avg", "Hollow_Wear_Diff",
    "qR_Avg", "qR_Diff",
    "Flange_Width_Avg", "Flange_Width_Diff",
    "Flange_Angle_Avg", "Flange_Angle_Diff",
    "Flange Width (Sd)_Avg",
    "Flange Height (Sh)_Avg",
    "Max Hollow Wear_Avg",
    "Flange Radius_Avg",
    "Min Flange Root Radius_Avg"
]

group_cols = ["Truck_ID", "Rail_Profile"]

# Confirm hunting outcomes agree across the two axles
assert (
    truck_df.groupby(group_cols)["Empty_Truck_Hunted"]
    .nunique(dropna=False).eq(1).all()
)

truck_df = (
    truck_df.groupby(group_cols)
    .agg({
        **{feature: "mean" for feature in features},
        "Empty_Truck_Hunted": "first"
    })
    .reset_index()
)

In [ ]:
# Add rail profile as a categorical predictor
X = truck_df[features + ["Rail_Profile"]].copy()
X = pd.get_dummies(
    X, columns=["Rail_Profile"], dtype=int
)

y = truck_df["Empty_Truck_Hunted"]
groups = truck_df["Truck_ID"]

print("Class distribution:")
print(y.value_counts())

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=4,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

cv = GroupKFold(n_splits=5)

# Out-of-fold predictions
y_pred = cross_val_predict(
    model,
    X,
    y,
    cv=cv,
    groups=groups
)

print(classification_report(y, y_pred, zero_division=0))
print(confusion_matrix(y, y_pred))

# Save the new model's values for use in the visual
engineered_y = y.copy()
engineered_pred = y_pred.copy()
engineered_keys = truck_df.loc[
    y.index, ["Truck_ID", "Rail_Profile"]
].reset_index(drop=True)

assert original_keys.equals(engineered_keys)
assert np.array_equal(original_y, engineered_y)

The expanded RF has lower accuracy, balanced accuracy and recall
for both classes under these truck-grouped folds. We did not run a significance
test here, and a lower point score does not show that each added measurement is
unhelpful. Notebook 08 repeats the comparison with complete cars held out;
those scores answer a different validation question.

In [ ]:
model.fit(X, y)

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importance.head(15).sort_values().plot(
    kind="barh",
    figsize=(9, 6)
)

plt.xlabel("Feature Importance")
plt.title("Expanded RF feature importance — fit to all assessments")
plt.tight_layout()
plt.show()


In [ ]:
def get_scores(y, pred):
    return [
        accuracy_score(y, pred),
        balanced_accuracy_score(y, pred),
        recall_score(y, pred, pos_label=True),
        recall_score(y, pred, pos_label=False)
    ]

comparison = pd.DataFrame({
    "Original RF": get_scores(original_y, original_pred),
    "Expanded RF": get_scores(engineered_y, engineered_pred)
}, index=[
    "Accuracy",
    "Balanced Accuracy",
    "Hunting Recall",
    "No-Hunting Recall"
])

ax = comparison.plot(kind="bar", figsize=(9, 5))

# Add values above each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)
    
plt.title("Random Forest Performance by Feature Set")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()


In [ ]:
plot_df = truck_df.copy()

# Average left and right flange width
plot_df["Flange_Mean"] = (
    plot_df["Flange Width (Sd)_L"] +
    plot_df["Flange Width (Sd)_R"]
) / 2

plt.figure(figsize=(9, 5))

for hunted, label in [(False, "No Hunting"), (True, "Hunting")]:
    subset = plot_df[
        plot_df["Empty_Truck_Hunted"] == hunted
    ]

    plt.scatter(
        subset["Flange_Mean"],
        subset["Eq Con 3mm"],
        label=label,
        alpha=0.7
    )

plt.xlabel("Truck mean flange width (Sd)")
plt.ylabel("Equivalent Conicity (3mm)")
plt.title("Flange Width vs Equivalent Conicity")
plt.legend()
plt.tight_layout()
plt.show()

The importance bars describe the expanded forest fitted to all
assessments; they are not held-out feature effects. Correlated original and
derived measurements can share importance. The scatter has one point per
truck/rail assessment, and conicity also changes with rail condition. These
plots help describe the fitted data without identifying a physical cause.